# Two-audio F0/PDA analysis

给定一段 Original 音频和一段 Reconstructed 音频，使用 Fig.2 一致的 F0 estimator、confidence threshold 和 cents tolerance，计算零时延的 frame-level PDA（Original ↔ Reconstructed）。

## Scope and metric contract

本 notebook 不需要 Human note annotation，因此只输出 **frame-level PDA**，而不是 Fig.2 Panel A 的 note-level PDA。两者的 estimator、F0 range、严格 confidence mask、PDA 分母和 cents criterion 保持一致：

- estimator：CREPE / PESTO / SwiftF0；
- confidence：Original 与 Reconstructed 都必须满足 `confidence > estimator-specific threshold`；
- PDA：只在两个轨迹均 confident 且 F0 有效的 frame 上计算；
- tolerance：`abs(1200 × log2(F0_reconstructed / F0_original)) ≤ tolerance`；
- 不做时延扫描或平移校正；仅以 0 s 对齐，并报告实际进入共同时间网格的 frame 数。

若需要 Fig.2 可直接比较的 note-level PDA，必须额外提供每个 note 的起止时间；不能把整段音频当成一个 note。

## 1. Setup

In [14]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# 支持从项目根目录、adaptation 或当前分析目录启动 notebook。
cwd = Path.cwd().resolve()
analysis_candidates = [cwd, cwd / 'Res_Ana_F0', cwd / 'adaptation' / 'Res_Ana_F0']
ANALYSIS_DIR = next(
    (candidate for candidate in analysis_candidates if (candidate / 'pitch_analysis_shared.py').is_file()),
    None,
)
if ANALYSIS_DIR is None:
    raise FileNotFoundError('无法定位 adaptation/Res_Ana_F0')
ANALYSIS_DIR = ANALYSIS_DIR.resolve()

if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from f0_backends import extract_f0
from pitch_analysis_shared import (
    F0Track,
    backend_configs,
    chroma_error_cents_from_signed,
    load_audio_mono,
    pitch_error_cents,
    shift_track_to_grid,
    valid_f0,
)


## 2. Input the two audio paths

In [15]:
# 唯一需要填写的内容：依次填入 [Original, Reconstructed] 两个音频的绝对路径。
AUDIO_PATHS = [
    '/data/zhj/neu2sou/neu_data/ZJM-260331-13/s2_voice_42.wav',
    '/data/zhj/neu2sou/neu_data/ZJM-260331-13/recon_s2_voice_42.flac',
]
if len(AUDIO_PATHS) != 2:
    raise ValueError('AUDIO_PATHS 必须且只能包含 [Original, Reconstructed] 两个路径。')
ORIGINAL_AUDIO_PATH, RECONSTRUCTED_AUDIO_PATH = map(Path, AUDIO_PATHS)

# 以下为固定的 Fig.2 指标口径；通常不需要修改。

# 与 Fig2_pitch_information_preserved.ipynb 一致的 estimator 和 estimator-native threshold。
ANALYSIS_SAMPLE_RATE = 48_000
ESTIMATOR_THRESHOLDS = {
    'crepe': 0.49512,
    'pesto': 0.19555,
    'swiftf0': 0.77354,
}
ESTIMATOR_LABELS = {'crepe': 'CREPE', 'pesto': 'PESTO', 'swiftf0': 'SwiftF0'}
PDA_TOLERANCES_CENTS = (200.0, 100.0, 50.0)  # Fig.2 clean tolerance ladder
PDA_FRAME_MASK = 'both'  # Fig.2 primary contract: both O/R must pass threshold
TIME_SHIFT_SEC = 0.0     # 固定零时延；不做时延对齐或 peak-shift 搜索。

if PDA_FRAME_MASK not in {'both', 'original'}:
    raise ValueError("PDA_FRAME_MASK must be 'both' or 'original'")
if not all(0.0 <= threshold <= 1.0 for threshold in ESTIMATOR_THRESHOLDS.values()):
    raise ValueError('All confidence thresholds must lie in [0, 1].')
if any(tolerance < 0 for tolerance in PDA_TOLERANCES_CENTS):
    raise ValueError('PDA tolerances must be non-negative.')
# for path in (ORIGINAL_AUDIO_PATH, RECONSTRUCTED_AUDIO_PATH):
#     if not path.is_file():
#         raise FileNotFoundError(f'请在参数单元填入存在的音频文件路径：{path}')

F0_CONFIGS = backend_configs(sr=ANALYSIS_SAMPLE_RATE)
parameter_table = pd.DataFrame([
    ('Metric', 'PDA (Original ↔ Reconstructed)'),
    ('Evaluation level', 'frame'),
    ('F0 range', f"{F0_CONFIGS['crepe']['fmin']:.0f}–{F0_CONFIGS['crepe']['fmax']:.0f} Hz"),
    ('Sample rate / nominal hop', f"{ANALYSIS_SAMPLE_RATE:,} Hz / {F0_CONFIGS['crepe']['hop_length'] / ANALYSIS_SAMPLE_RATE * 1_000:.1f} ms"),
    ('Confidence rule', 'strict confidence > estimator-specific threshold'),
    ('PDA frame mask', PDA_FRAME_MASK),
    ('Tolerance sweep', ' / '.join(f'±{value:g}¢' for value in PDA_TOLERANCES_CENTS)),
    ('Time policy', '0 s shift; nearest timestamp mapping only'),
], columns=['Parameter', 'Value'])
display(parameter_table)


,Parameter,Value
0,Metric,PDA (Original ↔ Reconstructed)
1,Evaluation level,frame
2,F0 range,150–750 Hz
3,Sample rate / nominal hop,"48,000 Hz / 5.0 ms"
4,Confidence rule,strict confidence > estimator-specific threshold
5,PDA frame mask,both
6,Tolerance sweep,±200¢ / ±100¢ / ±50¢
7,Time policy,0 s shift; nearest timestamp mapping only


## 3. Load audio and extract F0 trajectories

In [16]:
original_audio = load_audio_mono(ORIGINAL_AUDIO_PATH, target_sr=ANALYSIS_SAMPLE_RATE)
reconstructed_audio = load_audio_mono(RECONSTRUCTED_AUDIO_PATH, target_sr=ANALYSIS_SAMPLE_RATE)

audio_table = pd.DataFrame([
    {'audio': 'Original', 'path': str(ORIGINAL_AUDIO_PATH), 'duration_s': len(original_audio) / ANALYSIS_SAMPLE_RATE},
    {'audio': 'Reconstructed', 'path': str(RECONSTRUCTED_AUDIO_PATH), 'duration_s': len(reconstructed_audio) / ANALYSIS_SAMPLE_RATE},
])
display(audio_table.style.format({'duration_s': '{:.3f}'}))

def extract_pair_tracks(original_waveform, reconstructed_waveform, estimator):
    """Extract one same-estimator F0 pair using the shared project backend configuration."""
    config = F0_CONFIGS[estimator]
    original_f0, original_confidence, original_time = extract_f0(
        original_waveform, ANALYSIS_SAMPLE_RATE, estimator, config
    )
    reconstructed_f0, reconstructed_confidence, reconstructed_time = extract_f0(
        reconstructed_waveform, ANALYSIS_SAMPLE_RATE, estimator, config
    )
    return {
        'reference': F0Track(original_f0, original_confidence, original_time, estimator),
        'reconstructed': F0Track(reconstructed_f0, reconstructed_confidence, reconstructed_time, estimator),
    }

tracks_by_estimator = {
    estimator: extract_pair_tracks(original_audio, reconstructed_audio, estimator)
    for estimator in ESTIMATOR_THRESHOLDS
}

trajectory_table = pd.DataFrame([
    {
        'estimator': ESTIMATOR_LABELS[estimator],
        'Original frames': len(track_pair['reference'].time_sec),
        'Reconstructed frames': len(track_pair['reconstructed'].time_sec),
        'Original duration (s)': track_pair['reference'].time_sec[-1],
        'Reconstructed duration (s)': track_pair['reconstructed'].time_sec[-1],
    }
    for estimator, track_pair in tracks_by_estimator.items()
])
display(trajectory_table.style.format({'Original duration (s)': '{:.3f}', 'Reconstructed duration (s)': '{:.3f}'}))


,audio,path,duration_s
0,Original,/data/zhj/neu2sou/neu_data/ZJM-260331-13/s2_voice_42.wav,1.000
1,Reconstructed,/data/zhj/neu2sou/neu_data/ZJM-260331-13/recon_s2_voice_42.flac,1.000


,estimator,Original frames,Reconstructed frames,Original duration (s),Reconstructed duration (s)
0,CREPE,201,201,1.000,1.000
1,PESTO,201,201,1.000,1.000
2,SwiftF0,62,62,0.984,0.984


## 4. Zero-lag frame-level PDA

In [17]:
def zero_lag_align_reconstruction(reference_track, reconstructed_track):
    """Map Reconstruction onto the Original timestamp grid at exactly 0 s shift.

    This is timestamp-grid mapping, not delay correction: a frame is retained only
    when its nearest reconstructed timestamp is within half a reconstructed hop.
    """
    if len(reference_track.time_sec) < 2 or len(reconstructed_track.time_sec) < 2:
        raise ValueError('Both F0 tracks need at least two timestamps for frame-level PDA.')
    aligned_reconstruction = shift_track_to_grid(
        reference_track.time_sec, reconstructed_track, shift_sec=TIME_SHIFT_SEC
    )
    candidate_time = np.asarray(reconstructed_track.time_sec, dtype=float)
    reference_time = np.asarray(reference_track.time_sec, dtype=float)
    right = np.searchsorted(candidate_time, reference_time, side='left')
    left = np.clip(right - 1, 0, len(candidate_time) - 1)
    right = np.clip(right, 0, len(candidate_time) - 1)
    nearest = np.where(
        np.abs(candidate_time[left] - reference_time) <= np.abs(candidate_time[right] - reference_time),
        left,
        right,
    )
    half_hop = np.median(np.diff(candidate_time)) / 2.0 + 1e-9
    matched_timestamp = np.abs(candidate_time[nearest] - reference_time) <= half_hop
    return aligned_reconstruction, matched_timestamp

def safe_rate(numerator, denominator):
    return float(numerator / denominator) if denominator else np.nan

def compute_frame_pda(reference_track, reconstructed_track, threshold, tolerance_cents, matched_timestamp):
    """Compute Fig.2-contract frame PDA for one estimator and one tolerance."""
    if len(reference_track.f0_hz) != len(reconstructed_track.f0_hz):
        raise AssertionError('Aligned F0 tracks must have equal length.')
    matched_timestamp = np.asarray(matched_timestamp, dtype=bool)
    original_valid = valid_f0(reference_track.f0_hz)
    reconstructed_valid = valid_f0(reconstructed_track.f0_hz)
    original_confident = (
        matched_timestamp & original_valid & np.isfinite(reference_track.confidence)
        & (reference_track.confidence > threshold)
    )
    reconstructed_confident = (
        matched_timestamp & reconstructed_valid & np.isfinite(reconstructed_track.confidence)
        & (reconstructed_track.confidence > threshold)
    )
    if PDA_FRAME_MASK == 'both':
        pda_evaluation_mask = original_confident & reconstructed_confident
    else:
        # Fig.2 shared contract: low reconstructed confidence is ignored in this mode,
        # but an invalid reconstructed F0 remains in the PDA denominator as an error.
        pda_evaluation_mask = original_confident
    pda_valid_pair = pda_evaluation_mask & reconstructed_valid
    signed_cents = np.full(len(reference_track.f0_hz), np.nan, dtype=float)
    signed_cents[pda_valid_pair] = pitch_error_cents(
        reference_track.f0_hz[pda_valid_pair], reconstructed_track.f0_hz[pda_valid_pair]
    )
    absolute_cents = np.abs(signed_cents)
    chroma_cents = chroma_error_cents_from_signed(signed_cents)
    n_correct = int((pda_valid_pair & (absolute_cents <= tolerance_cents)).sum())
    n_chroma_correct = int((pda_valid_pair & (chroma_cents <= tolerance_cents)).sum())
    n_pda_evaluated = int(pda_evaluation_mask.sum())
    return {
        'n_reference_frames': int(len(reference_track.f0_hz)),
        'n_time_matched_frames': int(matched_timestamp.sum()),
        'n_original_confident': int(original_confident.sum()),
        'n_reconstructed_confident': int(reconstructed_confident.sum()),
        'n_original_reconstructed_voiced': int(pda_valid_pair.sum()),
        'n_pda_evaluated': n_pda_evaluated,
        'n_pda_correct': n_correct,
        'n_rca_pda_correct': n_chroma_correct,
        'pda': safe_rate(n_correct, n_pda_evaluated),
        'rca_pda': safe_rate(n_chroma_correct, n_pda_evaluated),
        'pda_candidate_recall': safe_rate(int(pda_valid_pair.sum()), int(original_confident.sum())),
        'joint_frame_coverage': safe_rate(n_pda_evaluated, int(matched_timestamp.sum())),
        'median_absolute_cents': float(np.nanmedian(absolute_cents)) if pda_valid_pair.any() else np.nan,
    }


## 5. PDA summary across estimator and tolerance

In [18]:
rows = []
alignment_rows = []
for estimator, track_pair in tracks_by_estimator.items():
    reference_track = track_pair['reference']
    aligned_reconstruction, matched_timestamp = zero_lag_align_reconstruction(
        reference_track, track_pair['reconstructed']
    )
    if not matched_timestamp.any():
        raise ValueError(f'{ESTIMATOR_LABELS[estimator]} has no zero-lag timestamp overlap between the two files.')
    alignment_rows.append({
        'estimator': ESTIMATOR_LABELS[estimator],
        'reference_grid_frames': len(reference_track.time_sec),
        'zero_lag_matched_frames': int(matched_timestamp.sum()),
        'zero_lag_grid_coverage': float(matched_timestamp.mean()),
    })
    for tolerance_cents in PDA_TOLERANCES_CENTS:
        row = compute_frame_pda(
            reference_track, aligned_reconstruction,
            threshold=ESTIMATOR_THRESHOLDS[estimator],
            tolerance_cents=tolerance_cents,
            matched_timestamp=matched_timestamp,
        )
        rows.append({
            'estimator': ESTIMATOR_LABELS[estimator],
            'backend': estimator,
            'confidence_threshold': ESTIMATOR_THRESHOLDS[estimator],
            'tolerance_cents': tolerance_cents,
            'tolerance_label': f'±{tolerance_cents:g}¢',
            'pda_frame_mask': PDA_FRAME_MASK,
            **row,
        })

pda_summary = pd.DataFrame(rows).sort_values(['tolerance_cents', 'estimator'], ascending=[False, True])
alignment_summary = pd.DataFrame(alignment_rows)

# 基本口径检查：rate 必须来自其显示的正确数与分母。
assert (pda_summary['n_pda_correct'] <= pda_summary['n_pda_evaluated']).all()
assert (pda_summary['n_rca_pda_correct'] <= pda_summary['n_pda_evaluated']).all()
assert pda_summary['pda'].dropna().between(0.0, 1.0).all()
assert pda_summary['rca_pda'].dropna().between(0.0, 1.0).all()

display(Markdown('### Timestamp-grid overlap (0 s shift)'))
display(alignment_summary.style.format({'zero_lag_grid_coverage': '{:.1%}'}))

display(Markdown('### PDA (%) by estimator and tolerance'))
pda_pivot = pda_summary.pivot(
    index=['estimator', 'confidence_threshold'],
    columns='tolerance_label',
    values='pda',
).reindex(columns=[f'±{value:g}¢' for value in PDA_TOLERANCES_CENTS])
display(pda_pivot.style.format('{:.2%}'))

display(Markdown('### Full frame-level PDA accounting'))
display_columns = [
    'estimator', 'confidence_threshold', 'tolerance_label', 'pda', 'n_pda_correct',
    'n_pda_evaluated', 'n_original_reconstructed_voiced', 'pda_candidate_recall', 'joint_frame_coverage',
    'rca_pda', 'median_absolute_cents',
]
display(pda_summary[display_columns].style.format({
    'confidence_threshold': '{:.5f}',
    'pda': '{:.2%}',
    'pda_candidate_recall': '{:.2%}',
    'joint_frame_coverage': '{:.2%}',
    'rca_pda': '{:.2%}',
    'median_absolute_cents': '{:.1f}',
}))


### Timestamp-grid overlap (0 s shift)

,estimator,reference_grid_frames,zero_lag_matched_frames,zero_lag_grid_coverage
0,CREPE,201,201,100.0%
1,PESTO,201,201,100.0%
2,SwiftF0,62,62,100.0%


### PDA (%) by estimator and tolerance

,tolerance_label,±200¢,±100¢,±50¢
estimator,confidence_threshold,,,
CREPE,0.495120,7.80%,7.80%,2.13%
PESTO,0.195550,7.89%,4.61%,1.97%
SwiftF0,0.773540,0.00%,0.00%,0.00%


### Full frame-level PDA accounting

,estimator,confidence_threshold,tolerance_label,pda,n_pda_correct,n_pda_evaluated,n_original_reconstructed_voiced,pda_candidate_recall,joint_frame_coverage,rca_pda,median_absolute_cents
0,CREPE,0.49512,±200¢,7.80%,11,141,141,89.81%,70.15%,14.18%,371.0
3,PESTO,0.19555,±200¢,7.89%,12,152,152,96.82%,75.62%,12.50%,386.1
6,SwiftF0,0.77354,±200¢,0.00%,0,2,2,4.35%,3.23%,0.00%,686.1
1,CREPE,0.49512,±100¢,7.80%,11,141,141,89.81%,70.15%,9.22%,371.0
4,PESTO,0.19555,±100¢,4.61%,7,152,152,96.82%,75.62%,4.61%,386.1
7,SwiftF0,0.77354,±100¢,0.00%,0,2,2,4.35%,3.23%,0.00%,686.1
2,CREPE,0.49512,±50¢,2.13%,3,141,141,89.81%,70.15%,2.13%,371.0
5,PESTO,0.19555,±50¢,1.97%,3,152,152,96.82%,75.62%,1.97%,386.1
8,SwiftF0,0.77354,±50¢,0.00%,0,2,2,4.35%,3.23%,0.00%,686.1


## Interpretation notes

- `PDA` 是在 **both-confident** frame 上的条件准确率；必须与 `pda_candidate_recall` 和 `joint_frame_coverage` 一起阅读。
- 各 estimator 的 confidence 并未跨方法校准，因此相同阈值语义不能跨 estimator 直接等价；本 notebook 使用 Fig.2 的 estimator-native operating points。
- 若音频存在真实固定时延，本结果会如实反映该时延造成的逐 frame 不匹配；本 notebook 不通过移动轨迹来提高 PDA。
- 本 notebook 不写入项目 F0 cache 或分析结果文件；`pda_summary` 和 `alignment_summary` 保留在内存中，可由后续单元按需导出。